In [ ]:
# ========================================================
# Week 15 : Decoder-only Transformer
# Character-level Text Generation (Shakespeare)
# ========================================================
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

tf.random.set_seed(42)
np.random.seed(42)

SEQ_LEN     = 30
EMBED_DIM   = 32
EPOCHS      = 20
BATCH       = 32
NUM_SAMPLES = 5000
MAX_TOKENS  = 10000

# =========================================
# 1. Load Shakespeare
# =========================================
path = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)
text = open(path, 'r').read()[:MAX_TOKENS]
print(f"Text length: {len(text):,} chars")

chars       = sorted(set(text))
VOCAB_SIZE  = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
encoded     = np.array([char_to_idx[c] for c in text])
print(f"Unique chars: {VOCAB_SIZE}")

# --- Sequence pairs (Many-to-Many for Transformer) ---
X, y = [], []
for i in range(len(encoded) - SEQ_LEN):
    X.append(encoded[i:i+SEQ_LEN])
    y.append(encoded[i+1:i+SEQ_LEN+1])
X = np.array(X[:NUM_SAMPLES])
y = np.array(y[:NUM_SAMPLES])
print(f"Sequences: {X.shape}")

# =========================================
# 2. Train / Val / Test Split  (60 / 20 / 20)
# =========================================
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)
print(f"Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}  |  Test: {X_test.shape[0]}")

# =========================================
# 3. Positional Encoding
# =========================================
def get_positional_encoding(seq_len, d_model):
    pos    = np.arange(seq_len)[:, np.newaxis]
    dim    = np.arange(d_model)[np.newaxis, :]
    angles = pos / np.power(10000, (2 * (dim // 2)) / d_model)
    pe     = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(pe[np.newaxis, :, :], dtype=tf.float32)

# =========================================
# 4. Model Definition — Decoder-only Transformer
# =========================================
class DecoderTransformer(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM)
        self.pe        = get_positional_encoding(SEQ_LEN, EMBED_DIM)
        self.mha       = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=EMBED_DIM)
        self.norm1     = tf.keras.layers.LayerNormalization()
        self.ffn1      = tf.keras.layers.Dense(128, activation='relu')
        self.ffn2      = tf.keras.layers.Dense(EMBED_DIM)
        self.norm2     = tf.keras.layers.LayerNormalization()
        self.out       = tf.keras.layers.Dense(VOCAB_SIZE, activation='softmax')

    def call(self, x, return_attention=False):
        seq_len = tf.shape(x)[1]
        x = self.embedding(x) + self.pe[:, :seq_len, :]
        attn_out, attn_scores = self.mha(
            x, x, x, use_causal_mask=True, return_attention_scores=True)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn2(self.ffn1(x)))
        logits = self.out(x)
        return (logits, attn_scores) if return_attention else logits

model_tf = DecoderTransformer()
model_tf.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# =========================================
# 5. Train (validation_data = Val set)
# =========================================
print("\nTraining: Decoder Transformer")
h_tf = model_tf.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH, verbose=1
)



# =========================================
# 6. Final Test Evaluation (1회만)
# =========================================
test_loss, test_acc = model_tf.evaluate(X_test, y_test, verbose=0)
print(f"\nTransformer — Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

# =========================================
# 7. Loss & Accuracy (Train / Val subplot)
# =========================================
ep = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Train / Val Loss
axes[0].plot(ep, h_tf.history['loss'],     'r-o', label='Train Loss')
axes[0].plot(ep, h_tf.history['val_loss'], 'b--s', label='Val Loss')
axes[0].set(title='Transformer — Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(True, alpha=.3)

# Train / Val Accuracy
axes[1].plot(ep, h_tf.history['accuracy'],     'r-o', label='Train Accuracy')
axes[1].plot(ep, h_tf.history['val_accuracy'], 'b--s', label='Val Accuracy')
axes[1].set(title='Transformer — Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=.3)

plt.suptitle('Decoder-only Transformer — Train / Validation Curves', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


# =========================================
# 8. Final Bar Chart (Test Accuracy)
# =========================================
fig, ax = plt.subplots(figsize=(4, 4))
bar = ax.bar(['Decoder\nTransformer'], [test_acc], color='#3498db', edgecolor='white', width=0.3)
ax.text(bar[0].get_x() + bar[0].get_width()/2, test_acc + 0.005, f'{test_acc:.4f}',
        ha='center', fontweight='bold', fontsize=13)
ax.set(title='Shakespeare — Decoder Transformer (Test Set)', ylabel='Test Accuracy')
ax.set_ylim(test_acc - 0.05, test_acc + 0.04)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

# =========================================
# 9. Attention Heatmap
# =========================================
sample       = X_test[:1]
_, attn      = model_tf(sample, return_attention=True)
sample_chars = [idx_to_char[i] for i in sample[0]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, cmap, head_idx, title in [
    (ax1, 'Blues', 0, 'Head 0'),
    (ax2, 'Reds',  1, 'Head 1')
]:
    scores = attn[0, head_idx].numpy()
    im     = ax.imshow(scores, cmap=cmap)
    step   = max(1, len(sample_chars) // 15)
    ticks  = range(0, len(sample_chars), step)
    labels = [sample_chars[i] for i in ticks]
    ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticks(ticks); ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle('Decoder Transformer — Causal Self-Attention', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

# =========================================
# 10. Temperature Generation
# =========================================
def generate_tf(seed, length=200, temperature=1.0):
    result    = list(seed)
    input_seq = [char_to_idx[c] for c in seed[-SEQ_LEN:]]
    for _ in range(length):
        x        = np.array([input_seq[-SEQ_LEN:]])
        preds    = model_tf(x, return_attention=False).numpy()[0, -1]
        preds    = np.exp(np.log(preds + 1e-8) / temperature)
        preds   /= preds.sum()
        next_idx = np.random.choice(len(preds), p=preds)
        result.append(idx_to_char[next_idx])
        input_seq.append(next_idx)
    return ''.join(result[len(seed):])

seed         = text[:SEQ_LEN]
temperatures = [0.2, 0.5, 1.0]

for t in temperatures:
    print(f"\n{'='*60}\n Temperature = {t}\n{'='*60}")
    print(f"\n[Decoder Transformer]\n{generate_tf(seed, 200, t)}")




Text length: 10,000 chars
Unique chars: 57
Sequences: (5000, 30)
Train: 3000  |  Val: 1000  |  Test: 1000

Training: Decoder Transformer
Epoch 1/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 48ms/step - accuracy: 0.1488 - loss: 3.3605 - val_accuracy: 0.2177 - val_loss: 2.9848
Epoch 2/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.2814 - loss: 2.6643 - val_accuracy: 0.3180 - val_loss: 2.4246
Epoch 3/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.3310 - loss: 2.3387 - val_accuracy: 0.3452 - val_loss: 2.2713
Epoch 4/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - accuracy: 0.3482 - loss: 2.2277 - val_accuracy: 0.3620 - val_loss: 2.1911
Epoch 5/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.3676 - loss: 2.1506 - val_accuracy: 0.3811 - val_loss: 2.1196
Epoch 6/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.3911 - loss: 2.0754 - val_accuracy: 0.4040 - val_loss: 2.0443
Epoch 7/20
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.4095 - loss: 2.0003 - val_accuracy: 